In [1]:
%reload_ext autoreload
%autoreload 2
# notebooks/result_processing.ipynb

# ---
# # Experiment Results Analysis
# This notebook orchestrates the loading, analysis, validation, and reporting
# of experiment results logged to Weights & Biases.
# ---

# ## 1. Setup and Imports
# Import necessary libraries and custom modules. The `sys.path.append` line
# allows the notebook to find the custom `picid_report` package.
import sys
import pandas as pd

sys.path.append("..")

from picid_report.run_processor import load_runs_df
from picid_report.analysis import analyze_results
from picid_report.reporting import (
    create_summary_table,
    display_experiment_stats,
    display_hp_impact,
)
import picid_report.validators as validators
import picid_report.config as config
import os

# ---
# ## 2. Configuration
# Define all key parameters for the analysis in one place.
# ---
PROJECT_NAME = "29_01_2026_nb14_prognostics_combined"
USER = "imos-buildingenergy-1"

# Metrics to extract
METRIC_PREFIXES = ["val/", "test/", "test_best_rerun/", "val_best_rerun/"]
REPORTING_METRICS = ["loss", "mse", "mae", "rmse", "f1", "accuracy"]

# --- NEW: Scientific Rigor Settings ---
# If defined, the analysis will DROP any configuration group that does not
# contain exactly these seeds. Set to `None` to include all available data.

# Data seeds are set to None (Flexible mode)
REQUIRED_DATA_SEEDS = None

# Model seeds are strictly enforced (Strict mode)
REQUIRED_MODEL_SEEDS = {72, 88, 101, 666, 226688}

# Columns to ignore if they vary within a group (avoids "Hidden Variation" warnings)
ADDITIONAL_IGNORED_COLS = ["gpu_id", "slurm_job_id"]

# Configure pandas display options for better readability in the notebook.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# ---
# ## 3. Data Loading
# Fetch data from wandb (or a local cache) and prepare the initial DataFrame.
# The `load_runs_df` function handles all the complex cleaning and flattening.
# ---
print("🚀 Loading Run Data...")
df, config_columns, dropped_columns = load_runs_df(project_name=PROJECT_NAME, user=USER)
print(f"Loaded {len(df)} runs.")


# 2. PROACTIVE PREPROCESSING: Rename models and remove prefixes
def clean_and_rename_models(df):
    target_col = config.COLUMN_CONFIG["model_target"]
    type_col = "model.model_type"

    # Remove common prefix (e.g., 'picid.baselines.', 'picid.model.wrappers.')
    # This finds the common prefix among all unique model names
    unique_models = df[target_col].unique()
    if len(unique_models) > 1:
        prefix = os.path.commonprefix(list(unique_models))
        # Ensure we only strip up to the last dot if possible
        if "." in prefix:
            prefix = prefix.rsplit(".", 1)[0] + "."
        df[target_col] = df[target_col].str.replace(prefix, "", regex=False)

    # Distinguish Statistical Baselines (Linear vs Exponential)
    if type_col in df.columns:
        mask = df[type_col].notna()
        df.loc[mask, target_col] = (
            df.loc[mask, target_col] + " (" + df.loc[mask, type_col].astype(str) + ")"
        )

    return df


df = clean_and_rename_models(df)

validators.validate_schema(df, required_cols=config.REQUIRED_COLUMNS)

# ---
# ## 4. Core Analysis
# Run the main analysis function. This now includes:
# 1. Grouping by Model + Hyperparameters
# 2. Strict Seed Validation (Filtering incomplete groups)
# 3. Hidden Variation Detection (Warning about silent bugs)
# 4. Aggregation & Best Model Selection
# ---
print("\n🔬 Analyzing Results & Validating Consistency...")

all_results = analyze_results(
    df=df,
    config_columns=config_columns,
    dropped_columns=dropped_columns,
    reporting_metrics=REPORTING_METRICS,
    metric_prefixes=METRIC_PREFIXES,
    # --- New Validation Arguments ---
    required_data_seeds=REQUIRED_DATA_SEEDS,
    required_model_seeds=REQUIRED_MODEL_SEEDS,
    additional_ignored_cols=ADDITIONAL_IGNORED_COLS,
)

# ---
# ## 5. Reporting & Visualizations
# ---

# ### A. Experiment Statistics
# Shows how many configurations completed successfully and
# confirms which seeds were actually found in the data.
display_experiment_stats(all_results)

# ### B. Performance Leaderboards
# Displays a separate, clean pivot table for every metric found (MSE, MAE, etc.).
# display_performance_tables(all_results)

# ### C. Hyperparameter Impact Analysis
# Shows the top 10 configurations for each model, exposing exactly
# which hyperparameters led to the best results.
display_hp_impact(all_results)

# ### D. Legacy Summary Table
# A single flattened table containing all metrics (useful for copying to Excel/LaTeX).
print("\n📊 Full Summary Table:")
summary_df = create_summary_table(all_results)
display(summary_df)

🚀 Loading Run Data...
Loaded 582 runs from cached file: csv_files/29_01_2026_nb14_prognostics_combined.csv
Dropped 118 columns with >90% NaN values.
The columns were drropped: ['ckpt_path', 'profiler_cfg', 'subset_range', 'val_subset_ratio', 'optimizer_factory', 'scheduler_factory', 'test_subset_ratio', 'plots/test/pred_vs_target_unit_(16,)', 'plots/test/pred_vs_target_unit_(19,)', 'plots/test/pred_vs_target_unit_(24,)', 'plots/test/pred_vs_target_unit_(28,)', 'plots/test/pred_vs_target_unit_(6,)', 'plots/test/pred_vs_target_unit_(8,)', 'test/mae_(16,)_denormalized', 'test/mae_(16,)_normalized', 'test/mae_(19,)_denormalized', 'test/mae_(19,)_normalized', 'test/mae_(24,)_denormalized', 'test/mae_(24,)_normalized', 'test/mae_(28,)_denormalized', 'test/mae_(28,)_normalized', 'test/mae_(6,)_denormalized', 'test/mae_(6,)_normalized', 'test/mae_(8,)_denormalized', 'test/mae_(8,)_normalized', 'test/mape_(16,)_denormalized', 'test/mape_(16,)_normalized', 'test/mape_(19,)_denormalized', 'test/m

,Dataset,Model,Configs Completed,Seeds Found
0,NB14,baselines.patchtst_model.PatchTST_Forecaster,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
1,NB14,baselines.spacetimeformer_model.Spacetimeformer_Forecaster,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
2,NB14,baselines.crossformer_model.Crossformer_Forecaster,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
3,NB14,model.wrappers.statistical_models_wrapper.StatisticalBaselineWrapper (exponential),15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
4,NB14,model.wrappers.fit_predict_tabpfn_wrapper.FitPredictTabPFNWrapper,6,Data: ? | Model: [42]
5,NB14,baselines.tide_model.TiDE_Forecaster,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
6,NB14,model.wrappers.cnn1d_wrapper.CNN1D_Wrapper,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
7,NB14,model.wrappers.mlp_wrapper.MLPWrapper,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
8,NB14,model.wrappers.statistical_models_wrapper.StatisticalBaselineWrapper (linear),15,"Data: ? | Model: [72, 88, 101, 666, 226688]"
9,NB14,baselines.timeseries_transformer_model.Timeseries_Transformer_Forecaster,15,"Data: ? | Model: [72, 88, 101, 666, 226688]"



  HYPERPARAMETER IMPACT ANALYSIS

>>> baselines.patchtst_model.PatchTST_Forecaster on NB14
    (Sorted by optimization target: val/loss)


,optimization.lr,task_definition.seq_len,model.ts_in,val/loss,test/loss,test/mae_denormalized_mean,test/mae_normalized_mean,test/mape_denormalized_mean,test/mape_normalized_mean,test/mse_denormalized_mean,test/mse_normalized_mean,test/phm_score_denormalized_mean,test/phm_score_normalized_mean,test/rmse_denormalized_mean,test/rmse_normalized_mean,test_best_rerun/loss,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_denormalized_mean,test_best_rerun/mse_normalized_mean,test_best_rerun/phm_score_denormalized_mean,test_best_rerun/phm_score_normalized_mean,test_best_rerun/rmse_denormalized_mean,test_best_rerun/rmse_normalized_mean,val/mae_denormalized_mean,val/mae_normalized_mean,val/mape_denormalized_mean,val/mape_normalized_mean,val/mse_denormalized_mean,val/mse_normalized_mean,val/phm_score_denormalized_mean,val/phm_score_normalized_mean,val/rmse_denormalized_mean,val/rmse_normalized_mean,val_best_rerun/loss,val_best_rerun/mae_denormalized_mean,val_best_rerun/mae_normalized_mean,val_best_rerun/mape_denormalized_mean,val_best_rerun/mape_normalized_mean,val_best_rerun/mse_denormalized_mean,val_best_rerun/mse_normalized_mean,val_best_rerun/phm_score_denormalized_mean,val_best_rerun/phm_score_normalized_mean,val_best_rerun/rmse_denormalized_mean,val_best_rerun/rmse_normalized_mean
6,0.0005,10,10.0,0.0043 ± 0.0001 (n=5),0.0045 ± 0.0000 (n=5),65.8630 ± 0.6700 (n=5),0.0608 ± 0.0006 (n=5),3.5010 ± 0.2680 (n=5),3.5010 ± 0.2680 (n=5),7089.4036 ± 213.7164 (n=5),0.0060 ± 0.0002 (n=5),0.2349 ± 0.0028 (n=5),0.2349 ± 0.0028 (n=5),77.5613 ± 0.5809 (n=5),0.0716 ± 0.0005 (n=5),0.0045 ± 0.0000 (n=5),65.8630 ± 0.6700 (n=5),0.0608 ± 0.0006 (n=5),3.5010 ± 0.2680 (n=5),3.5010 ± 0.2680 (n=5),7089.4036 ± 213.7164 (n=5),0.0060 ± 0.0002 (n=5),0.2349 ± 0.0028 (n=5),0.2349 ± 0.0028 (n=5),77.5613 ± 0.5809 (n=5),0.0716 ± 0.0005 (n=5),59.3844 ± 0.9261 (n=5),0.0548 ± 0.0009 (n=5),2.7441 ± 0.1956 (n=5),2.7441 ± 0.1956 (n=5),5637.7305 ± 236.8623 (n=5),0.0048 ± 0.0002 (n=5),0.2665 ± 0.0055 (n=5),0.2665 ± 0.0055 (n=5),73.9168 ± 1.2187 (n=5),0.0682 ± 0.0011 (n=5),0.0042 ± 0.0001 (n=5),58.8825 ± 0.5464 (n=5),0.0544 ± 0.0005 (n=5),2.8455 ± 0.2527 (n=5),2.8455 ± 0.2527 (n=5),5559.0275 ± 138.9670 (n=5),0.0047 ± 0.0001 (n=5),0.2683 ± 0.0033 (n=5),0.2683 ± 0.0033 (n=5),73.1268 ± 0.7099 (n=5),0.0675 ± 0.0007 (n=5)
12,0.0001,50,50.0,0.0044 ± 0.0003 (n=5),0.0038 ± 0.0004 (n=5),55.6478 ± 2.6616 (n=5),0.0514 ± 0.0025 (n=5),3.2511 ± 0.5221 (n=5),3.2511 ± 0.5221 (n=5),4801.2233 ± 508.8124 (n=5),0.0041 ± 0.0004 (n=5),0.2217 ± 0.0083 (n=5),0.2217 ± 0.0083 (n=5),68.1047 ± 3.5666 (n=5),0.0629 ± 0.0033 (n=5),0.0038 ± 0.0004 (n=5),55.6478 ± 2.6616 (n=5),0.0514 ± 0.0025 (n=5),3.2511 ± 0.5221 (n=5),3.2511 ± 0.5221 (n=5),4801.2233 ± 508.8124 (n=5),0.0041 ± 0.0004 (n=5),0.2217 ± 0.0083 (n=5),0.2217 ± 0.0083 (n=5),68.1047 ± 3.5666 (n=5),0.0629 ± 0.0033 (n=5),60.8264 ± 1.9023 (n=5),0.0562 ± 0.0018 (n=5),3.5879 ± 0.2995 (n=5),3.5879 ± 0.2995 (n=5),5650.4156 ± 427.7740 (n=5),0.0048 ± 0.0004 (n=5),0.2374 ± 0.0150 (n=5),0.2374 ± 0.0150 (n=5),74.2082 ± 2.6655 (n=5),0.0685 ± 0.0025 (n=5),0.0042 ± 0.0003 (n=5),59.2432 ± 1.7844 (n=5),0.0547 ± 0.0016 (n=5),3.5034 ± 0.2682 (n=5),3.5034 ± 0.2682 (n=5),5380.0936 ± 367.1143 (n=5),0.0046 ± 0.0003 (n=5),0.2406 ± 0.0049 (n=5),0.2406 ± 0.0049 (n=5),72.4304 ± 2.4957 (n=5),0.0669 ± 0.0023 (n=5)
11,0.0001,10,10.0,0.0057 ± 0.0006 (n=5),0.0063 ± 0.0005 (n=5),77.4232 ± 2.7448 (n=5),0.0715 ± 0.0025 (n=5),3.9369 ± 0.2649 (n=5),3.9369 ± 0.2649 (n=5),9896.7639 ± 626.0991 (n=5),0.0084 ± 0.0005 (n=5),0.2128 ± 0.0092 (n=5),0.2128 ± 0.0092 (n=5),91.2471 ± 3.4424 (n=5),0.0842 ± 0.0032 (n=5),0.0063 ± 0.0005 (n=5),77.4232 ± 2.7448 (n=5),0.0715 ± 0.0025 (n=5),3.9369 ± 0.2649 (n=5),3.9369 ± 0.2649 (n=5),9896.7639 ± 626.0991 (n=5),0.0084 ± 0.0005 (n=5),0.2128 ± 0.0092 (n=5),0.2128 ± 0.0092 (n=5),91.2471 ± 3.4424 (n=5),0.0842 ± 0.0032 (n=5),69.5823 ± 3.9376 (


>>> baselines.spacetimeformer_model.Spacetimeformer_Forecaster on NB14
    (Sorted by optimization target: val/loss)


...

,task_definition.seq_len,task_definition.stride_train,val/loss,test/loss,test/mae_denormalized_mean,test/mae_normalized_mean,test/mape_denormalized_mean,test/mape_normalized_mean,test/mse_denormalized_mean,test/mse_normalized_mean,test/phm_score_denormalized_mean,test/phm_score_normalized_mean,test/rmse_denormalized_mean,test/rmse_normalized_mean,test_best_rerun/loss,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_denormalized_mean,test_best_rerun/mse_normalized_mean,test_best_rerun/phm_score_denormalized_mean,test_best_rerun/phm_score_normalized_mean,test_best_rerun/rmse_denormalized_mean,test_best_rerun/rmse_normalized_mean,val/mae_denormalized_mean,val/mae_normalized_mean,val/mape_denormalized_mean,val/mape_normalized_mean,val/mse_denormalized_mean,val/mse_normalized_mean,val/phm_score_denormalized_mean,val/phm_score_normalized_mean,val/rmse_denormalized_mean,val/rmse_normalized_mean,val_best_rerun/loss,val_best_rerun/mae_denormalized_mean,val_best_rerun/mae_normalized_mean,val_best_rerun/mape_denormalized_mean,val_best_rerun/mape_normalized_mean,val_best_rerun/mse_denormalized_mean,val_best_rerun/mse_normalized_mean,val_best_rerun/phm_score_denormalized_mean,val_best_rerun/phm_score_normalized_mean,val_best_rerun/rmse_denormalized_mean,val_best_rerun/rmse_normalized_mean
0,1,1,-,1.0000 ± 0.0000 (n=1),41.7993 ± 0.0000 (n=1),0.0386 ± 0.0000 (n=1),1.4071 ± 0.0000 (n=1),1.4071 ± 0.0000 (n=1),2861.7988 ± 0.0000 (n=1),0.0024 ± 0.0000 (n=1),0.2451 ± 0.0000 (n=1),0.2451 ± 0.0000 (n=1),52.7592 ± 0.0000 (n=1),0.0487 ± 0.0000 (n=1),-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
1,5,1,-,1.0000 ± 0.0000 (n=1),41.7673 ± 0.0000 (n=1),0.0386 ± 0.0000 (n=1),2.1174 ± 0.0000 (n=1),2.1174 ± 0.0000 (n=1),2941.9949 ± 0.0000 (n=1),0.0025 ± 0.0000 (n=1),0.2910 ± 0.0000 (n=1),0.2910 ± 0.0000 (n=1),52.3130 ± 0.0000 (n=1),0.0483 ± 0.0000 (n=1),-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
2,10,5,-,1.0000 ± 0.0000 (n=1),48.7525 ± 0.0000 (n=1),0.0450 ± 0.0000 (n=1),2.7551 ± 0.0000 (n=1),2.7551 ± 0.0000 (n=1),4052.2131 ± 0.0000 (n=1),0.0035 ± 0.0000 (n=1),0.2799 ± 0.0000 (n=1),0.2799 ± 0.0000 (n=1),59.1192 ± 0.0000 (n=1),0.0546 ± 0.0000 (n=1),-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
3,20,5,-,1.0000 ± 0.0000 (n=1),48.1991 ± 0.0000 (n=1),0.0445 ± 0.0000 (n=1),3.0071 ± 0.0000 (n=1),3.0071 ± 0.0000 (n=1),3802.6609 ± 0.0000 (n=1),0.0032 ± 0.0000 (n=1),0.2700 ± 0.0000 (n=1),0.2700 ± 0.0000 (n=1),57.5949 ± 0.0000 (n=1),0.0532 ± 0.0000 (n=1),-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
4,50,50,-,1.0000 ± 0.0000 (n=1),72.5398 ± 0.0000 (n=1),0.0670 ± 0.0000 (n=1),6.9988 ± 0.0000 (n=1),6.9988 ± 0.0000 (n=1),8562.4521 ± 0.0000 (n=1),0.0073 ± 0.0000 (n=1),0.1970 ± 0.0000 (n=1),0.1970 ± 0.0000 (n=1),86.9495 ± 0.0000 (n=1),0.0803 ± 0.0000 (n=1),-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
5,100,100,-,1.0000 ± 0.0000 (n=1),103.5101 ± 0.0000 (n=1),0.0956 ± 0.0000 (n=1),9.8268 ± 0.0000 (n=1),9.8268 ± 0.0000 (n=1),16679.6074 ± 0.0000 (n=1),0.0142 ± 0.0000 (n=1),0.1143 ± 0.0000 (n=1),0.1143 ± 0.0000 (n=1),119.5650 ± 0.0000 (n=1),0.1104 ± 0.0000 (n=1),-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-



>>> baselines.lstm_model.LSTM_Forecaster on NB14
    (Sorted by optimization target: val/loss)


,task_definition.seq_len,optimization.lr,val/loss,test/loss,test/mae_denormalized_mean,test/mae_normalized_mean,test/mape_denormalized_mean,test/mape_normalized_mean,test/mse_denormalized_mean,test/mse_normalized_mean,test/phm_score_denormalized_mean,test/phm_score_normalized_mean,test/rmse_denormalized_mean,test/rmse_normalized_mean,test_best_rerun/loss,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_denormalized_mean,test_best_rerun/mse_normalized_mean,test_best_rerun/phm_score_denormalized_mean,test_best_rerun/phm_score_normalized_mean,test_best_rerun/rmse_denormalized_mean,test_best_rerun/rmse_normalized_mean,val/mae_denormalized_mean,val/mae_normalized_mean,val/mape_denormalized_mean,val/mape_normalized_mean,val/mse_denormalized_mean,val/mse_normalized_mean,val/phm_score_denormalized_mean,val/phm_score_normalized_mean,val/rmse_denormalized_mean,val/rmse_normalized_mean,val_best_rerun/loss,val_best_rerun/mae_denormalized_mean,val_best_rerun/mae_normalized_mean,val_best_rerun/mape_denormalized_mean,val_best_rerun/mape_normalized_mean,val_best_rerun/mse_denormalized_mean,val_best_rerun/mse_normalized_mean,val_best_rerun/phm_score_denormalized_mean,val_best_rerun/phm_score_normalized_mean,val_best_rerun/rmse_denormalized_mean,val_best_rerun/rmse_normalized_mean
9,100,0.0010,0.0019 ± 0.0002 (n=5),0.0019 ± 0.0002 (n=5),41.2270 ± 2.1486 (n=5),0.0381 ± 0.0020 (n=5),2.6186 ± 0.1395 (n=5),2.6186 ± 0.1395 (n=5),2877.2120 ± 321.5396 (n=5),0.0025 ± 0.0003 (n=5),0.2547 ± 0.0155 (n=5),0.2547 ± 0.0155 (n=5),49.9535 ± 2.3129 (n=5),0.0461 ± 0.0021 (n=5),0.0019 ± 0.0002 (n=5),41.2270 ± 2.1486 (n=5),0.0381 ± 0.0020 (n=5),2.6186 ± 0.1395 (n=5),2.6186 ± 0.1395 (n=5),2877.2120 ± 321.5396 (n=5),0.0025 ± 0.0003 (n=5),0.2547 ± 0.0155 (n=5),0.2547 ± 0.0155 (n=5),49.9535 ± 2.3129 (n=5),0.0461 ± 0.0021 (n=5),37.8514 ± 2.7414 (n=5),0.0349 ± 0.0025 (n=5),2.1897 ± 0.2090 (n=5),2.1897 ± 0.2090 (n=5),2873.5661 ± 306.9954 (n=5),0.0024 ± 0.0003 (n=5),0.3174 ± 0.0240 (n=5),0.3174 ± 0.0240 (n=5),46.9656 ± 3.8908 (n=5),0.0434 ± 0.0036 (n=5),0.0017 ± 0.0001 (n=5),35.7047 ± 1.1237 (n=5),0.0330 ± 0.0010 (n=5),2.1263 ± 0.1063 (n=5),2.1263 ± 0.1063 (n=5),2460.7995 ± 88.0099 (n=5),0.0021 ± 0.0001 (n=5),0.3514 ± 0.0081 (n=5),0.3514 ± 0.0081 (n=5),44.6926 ± 1.9828 (n=5),0.0413 ± 0.0018 (n=5)
10,100,0.0005,0.0020 ± 0.0001 (n=5),0.0020 ± 0.0003 (n=5),43.0109 ± 1.7850 (n=5),0.0397 ± 0.0016 (n=5),2.9748 ± 0.4641 (n=5),2.9748 ± 0.4641 (n=5),3114.9517 ± 307.8053 (n=5),0.0027 ± 0.0003 (n=5),0.2487 ± 0.0309 (n=5),0.2487 ± 0.0309 (n=5),51.8657 ± 3.0145 (n=5),0.0479 ± 0.0028 (n=5),0.0020 ± 0.0003 (n=5),43.0109 ± 1.7850 (n=5),0.0397 ± 0.0016 (n=5),2.9748 ± 0.4641 (n=5),2.9748 ± 0.4641 (n=5),3114.9517 ± 307.8053 (n=5),0.0027 ± 0.0003 (n=5),0.2487 ± 0.0309 (n=5),0.2487 ± 0.0309 (n=5),51.8657 ± 3.0145 (n=5),0.0479 ± 0.0028 (n=5),40.3497 ± 2.0700 (n=5),0.0372 ± 0.0019 (n=5),2.2622 ± 0.1026 (n=5),2.2622 ± 0.1026 (n=5),3122.5395 ± 235.7783 (n=5),0.0027 ± 0.0002 (n=5),0.3234 ± 0.0142 (n=5),0.3234 ± 0.0142 (n=5),49.9243 ± 2.3629 (n=5),0.0461 ± 0.0022 (n=5),0.0018 ± 0.0002 (n=5),38.3474 ± 2.3482 (n=5),0.0354 ± 0.0022 (n=5),2.3891 ± 0.2242 (n=5),2.3891 ± 0.2242 (n=5),2643.0551 ± 256.7642 (n=5),0.0023 ± 0.0002 (n=5),0.3283 ± 0.0214 (n=5),0.3283 ± 0.0214 (n=5),47.3626 ± 2.9052 (n=5),0.0437 ± 0.0027 (n=5)
6,50,0.0010,0.0024 ± 0.0004 (n=5),0.0020 ± 0.0002 (n=5),42.5443 ± 3.0809 (n=5),0.0393 ± 0.0028 (n=5),2.2734 ± 0.4509 (n=5),2.2734 ± 0.4509 (n=5),3011.6648 ± 367.6247 (n=5),0.0026 ± 0.0003 (n=5),0.2624 ± 0.0182 (n=5),0.2624 ± 0.0182 (n=5),51.3197 ± 3.1110 (n=5),0.0474 ± 0.0029 (n=5),0.0020 ± 0.0002 (n=5),42.5443 ± 3.0809 (n=5),0.0393 ± 0.0028 (n=5),2.2734 ± 0.4509 (n=5),2.2734 ± 0.4509 (n=5),3011.6648 ± 367.6247 (n=5),0.0026 ± 0.0003 (n=5),0.2624 ± 0.0182 (n=5),0.2624 ± 0.0182 (n=5),51.3197 ± 3.1110 (n=5),0.0474 ± 0.0029 (n=5),39.4864 ± 4.2683 (n=5),0.0365 ± 0.0039 (n=5),


📊 Full Summary Table:


Dataset                                                              NB14                                                                                                            \
Metric                                                          test/loss test/mae_denormalized_mean test/mae_normalized_mean test/mape_denormalized_mean test/mape_normalized_mean   
Model                                                                                                                                                                                 
baselines.crossformer_model.Crossformer_Forecaster  0.0018 ± 0.0003 (n=5)     41.8399 ± 3.6838 (n=5)    0.0386 ± 0.0034 (n=5)       2.8595 ± 0.4221 (n=5)     2.8595 ± 0.4221 (n=5)   
baselines.lstm_model.LSTM_Forecaster                0.0019 ± 0.0002 (n=5)     41.2270 ± 2.1486 (n=5)    0.0381 ± 0.0020 (n=5)       2.6186 ± 0.1395 (n=5)     2.6186 ± 0.1395 (n=5)   
baselines.patchtst_model.PatchTST_Forecaster        0.0045 ± 0.0000 (n=5)     65.8630 ± 0.6700 (n=5)    0.0608 ± 0.0006 (n=5)       3.5010 ± 0.2680 (n=5)     3.5010 ± 0.2680 (n=5)   
baselines.spacetimeformer_model.Spacetimeformer...  0.0053 ± 0.0017 (n=5)     68.0611 ± 8.7032 (n=5)    0.0628 ± 0.0080 (n=5)       5.6071 ± 2.1299 (n=5)     5.6071 ± 2.1299 (n=5)   
baselines.tide_model.TiDE_Forecaster                0.0014 ± 0.0002 (n=5)     36.3731 ± 1.1665 (n=5)    0.0336 ± 0.0011 (n=5)       2.6982 ± 0.4399 (n=5)     2.6982 ± 0.4399 (n=5)   
baselines.timeseries_transformer_model.Timeseri...  0.0040 ± 0.0007 (n=5)     63.1397 ± 5.7433 (n=5)    0.0583 ± 0.0053 (n=5)       4.1653 ± 0.1403 (n=5)     4.1653 ± 0.1403 (n=5)   
model.wrappers.cnn1d_wrapper.CNN1D_Wrapper          0.0131 ± 0.0059 (n=5)    98.9557 ± 15.4816 (n=5)    0.0914 ± 0.0143 (n=5)       6.1727 ± 1.1110 (n=5)     6.1727 ± 1.1110 (n=5)   
model.wrappers.fit_predict_tabdpt_wrapper.FitPr...  1.0000 ± 0.0000 (n=1)     41.7993 ± 0.0000 (n=1)    0.0386 ± 0.0000 (n=1)       1.4071 ± 0.0000 (n=1)     1.4071 ± 0.0000 (n=1)   
model.wrappers.fit_predict_tabpfn_wrapper.FitPr...  1.0000 ± 0.0000 (n=1)     41.7476 ± 0.0000 (n=1)    0.0385 ± 0.0000 (n=1)       1.4111 ± 0.0000 (n=1)     1.4111 ± 0.0000 (n=1)   
model.wrappers.mlp_wrapper.MLPWrapper               0.0510 ± 0.0571 (n=5)   161.9690 ± 96.3975 (n=5)    0.1495 ± 0.0890 (n=5)       9.7809 ± 4.6053 (n=5)     9.7809 ± 4.6053 (n=5)   
model.wrappers.statistical_models_wrapper.Stati...  0.0260 ± 0.0000 (n=5)    130.1766 ± 0.0116 (n=5)    0.1202 ± 0.0000 (n=5)       0.9991 ± 0.0005 (n=5)     0.9991 ± 0.0005 (n=5)   
model.wrappers.statistical_models_wrapper.Stati...  0.3216 ± 0.1614 (n=5)  417.9388 ± 107.8377 (n=5)    0.3858 ± 0.0996 (n=5)     35.4759 ± 16.7702 (n=5)   35.4759 ± 16.7702 (n=5)   

Dataset                                                                                                                                                                       \
Metric                                                   test/mse_denormalized_mean test/mse_normalized_mean test/phm_score_denormalized_mean test/phm_score_normalized_mean   
Model                                                                                                                                                                          
baselines.crossformer_model.Crossformer_Forecaster       3142.0465 ± 806.7998 (n=5)    0.0027 ± 0.0007 (n=5)            0.3083 ± 0.0163 (n=5)          0.3083 ± 0.0163 (n=5)   
baselines.lstm_model.LSTM_Forecaster                     2877.2120 ± 321.5396 (n=5)    0.0025 ± 0.0003 (n=5)            0.2547 ± 0.0155 (n=5)          0.2547 ± 0.0155 (n=5)   
baselines.patchtst_model.PatchTST_Forecaster             7089.4036 ± 213.7164 (n=5)    0.0060 ± 0.0002 (n=5)            0.2349 ± 0.0028 (n=5)          0.2349 ± 0.0028 (n=5)   
baselines.spacetimeformer_model.Spacetimeformer...      7230.2074 ± 1637.3606 (n=5)    0.0062 ± 0.0014 (n=5)            0.2029 ± 0.0310 (n=5)          0.2029 ± 0.0310 (n=5)   
baselines.tide_model.T

In [2]:
# 1. Access the raw data for the specific model and dataset
model_key = "picid.baselines.spacetimeformer_model.Spacetimeformer_Forecaster"
dataset_key = "NB14"

# This DF contains all runs for this model before aggregation
df_subset = all_results[dataset_key][model_key]["non_aggregated_df"]

In [5]:
[i for i in list(df_subset.columns) if "datasource" in i]

['datasource._target_',
 'datasource.data_dir',
 'datasource.data_name',
 'datasource.task_mode']

In [22]:
# 1. Access the raw data for the specific model and dataset
model_key = "picid.baselines.spacetimeformer_model.Spacetimeformer_Forecaster"
dataset_key = "UNIBO21"

# This DF contains all runs for this model before aggregation
df_subset = all_results[dataset_key][model_key]["non_aggregated_df"]

# 2. Filter for ONE specific configuration that shows count=15
# (Based on your screenshot: lr=0.0001, seq_len=10, batch_size=16)
mask = (
    (df_subset["optimization.lr"] == 0.0001)
    & (df_subset["task_definition.seq_len"] == 10)
    & (df_subset["datamodule.train_batch_size"] == 16)
)
group_runs = df_subset[mask]

# 3. Display identifiers to spot the duplicates
# 'created_at' will show you if these runs are from different times/days
cols_to_inspect = [
    "run_name",
    "id",
    "created_at",
    "seed",
    "datasource.parameters.data_seed",
]
valid_cols = [c for c in cols_to_inspect if c in group_runs.columns]

print(f"Found {len(group_runs)} runs for this configuration (Expected 5).")
print("Listing all runs in this group:")
# display(group_runs[valid_cols].sort_values('created_at'))

Found 5 runs for this configuration (Expected 5).
Listing all runs in this group:


In [ ]:
df_subset

\begin{tabular}{ll}
\toprule
 & 0 \\
\midrule
AvgTime/train_batch_mean & 0.020657 \\
AvgTime/train_batch_std & 0.005162 \\
AvgTime/train_epoch_mean & 11.607952 \\
AvgTime/train_epoch_std & 0.450640 \\
AvgTime/val_batch_mean & 0.008616 \\
AvgTime/val_batch_std & 0.000965 \\
AvgTime/val_epoch_mean & 0.036777 \\
AvgTime/val_epoch_std & 0.002732 \\
_runtime & 306 \\
_step & 490 \\
_timestamp & 1769624335.062802 \\
epoch & 25 \\
lr-AdamW & 0.000902 \\
plots/test/pred_vs_target_unit_10 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_10_478_79be3a6b41f2384b3f28.plotly.json', 'sha256': '79be3a6b41f2384b3f284f3a04ea9ec558be424cd78f12632c04666c0f8665d2', 'size': 29913} \\
plots/test/pred_vs_target_unit_11 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_11_485_aa7b2f092bbec702f9dc.plotly.json', 'sha256': 'aa7b2f092bbec702f9dcde687afc921f046ea5c1f5c4e71eeb175f1442ced9cc', 'size': 29299} \\
plots/test/pred_vs_target_unit_13 & {'_type':

In [19]:
print(df_subset.iloc[0].to_latex())

\begin{tabular}{ll}
\toprule
 & 30 \\
\midrule
AvgTime/train_batch_mean & 0.024092 \\
AvgTime/train_batch_std & 0.006863 \\
AvgTime/train_epoch_mean & 12.582578 \\
AvgTime/train_epoch_std & 1.215955 \\
AvgTime/val_batch_mean & 0.018669 \\
AvgTime/val_batch_std & 0.008642 \\
AvgTime/val_epoch_mean & 0.073752 \\
AvgTime/val_epoch_std & 0.012027 \\
_runtime & 1358 \\
_step & 1918 \\
_timestamp & 1762901279.126953 \\
epoch & 100 \\
lr-Adam & NaN \\
plots/test/pred_vs_target_unit_10 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_10_1906_ce34dedb41d76ea75180.plotly.json', 'sha256': 'ce34dedb41d76ea75180778ff341f8960afc2147b56aaece910ba30348b5591a', 'size': 28982} \\
plots/test/pred_vs_target_unit_11 & {'_type': 'plotly-file', 'path': 'media/plotly/plots/test/pred_vs_target_unit_11_1913_09482285d5caa3b46983.plotly.json', 'sha256': '09482285d5caa3b469839a8d5678ab7aeeee53c83d349bc317ef18a2e1c4c76e', 'size': 28379} \\
plots/test/pred_vs_target_unit_13 & {'_type':

In [12]:
import pandas as pd

# 1. Isolate the specific group
target_model = "picid.baselines.spacetimeformer_model.Spacetimeformer_Forecaster"

# Update this mask to match the specific "duplicate" group you saw in your table
mask = (
    (df["model._target_"] == target_model)
    & (df["optimization.lr"] == 0.0001)
    & (df["task_definition.seq_len"] == 10)
    & (df["datamodule.train_batch_size"] == 16)
)
group_runs = df[mask].copy()

# 2. Find "Hidden Variables" (Robust to lists/dicts)
varying_cols = []

for col in group_runs.columns:
    try:
        # Standard check for hashable types (int, float, str)
        if group_runs[col].nunique() > 1:
            varying_cols.append(col)
    except TypeError:
        # Fallback for unhashable types (lists, dicts): Convert to string first
        if group_runs[col].astype(str).nunique() > 1:
            varying_cols.append(col)

# 3. Add identifiers for context
# 'created_at' or '_timestamp' is CRITICAL to see if these are old vs new runs
context_cols = ["run_name", "seed", "created_at", "_timestamp"]

for c in context_cols:
    if c in group_runs.columns and c not in varying_cols:
        varying_cols.insert(0, c)

# 4. Display sorted by Seed to compare side-by-side
print(f"Found {len(group_runs)} runs in this group.")
print("Showing ONLY columns that differ between these runs:")

if not varying_cols:
    print("No varying columns found! These rows are EXACT duplicates.")
else:
    display(
        group_runs[varying_cols]
        .sort_values(
            by=["seed", "created_at"] if "created_at" in group_runs else ["seed"]
        )
        .style.set_properties(**{"text-align": "left"})
    )

Found 0 runs in this group.
Showing ONLY columns that differ between these runs:


,_timestamp,seed,run_name


In [6]:
group_runs.sort_values(
    by=["seed", "created_at"] if "created_at" in group_runs else ["seed"]
)["val_best_rerun/rmse_denormalized_mean"]

25    85.029541
26    88.634254
27    85.144997
28    88.849060
29    93.493942
Name: val_best_rerun/rmse_denormalized_mean, dtype: float64

In [7]:
group_runs.sort_values(
    by=["seed", "created_at"] if "created_at" in group_runs else ["seed"]
)[[i for i in group_runs.columns if "val_best_rerun" in i]]

,val_best_rerun/loss,val_best_rerun/mae_15_denormalized,val_best_rerun/mae_15_normalized,val_best_rerun/mae_18_denormalized,val_best_rerun/mae_18_normalized,val_best_rerun/mae_23_denormalized,val_best_rerun/mae_23_normalized,val_best_rerun/mae_27_denormalized,val_best_rerun/mae_27_normalized,val_best_rerun/mae_5_denormalized,val_best_rerun/mae_5_normalized,val_best_rerun/mae_7_denormalized,val_best_rerun/mae_7_normalized,val_best_rerun/mae_denormalized_mean,val_best_rerun/mae_normalized_mean,val_best_rerun/mape_15_denormalized,val_best_rerun/mape_15_normalized,val_best_rerun/mape_18_denormalized,val_best_rerun/mape_18_normalized,val_best_rerun/mape_23_denormalized,val_best_rerun/mape_23_normalized,val_best_rerun/mape_27_denormalized,val_best_rerun/mape_27_normalized,val_best_rerun/mape_5_denormalized,val_best_rerun/mape_5_normalized,val_best_rerun/mape_7_denormalized,val_best_rerun/mape_7_normalized,val_best_rerun/mape_denormalized_mean,val_best_rerun/mape_normalized_mean,val_best_rerun/mse_15_denormalized,val_best_rerun/mse_15_normalized,val_best_rerun/mse_18_denormalized,val_best_rerun/mse_18_normalized,val_best_rerun/mse_23_denormalized,val_best_rerun/mse_23_normalized,val_best_rerun/mse_27_denormalized,val_best_rerun/mse_27_normalized,val_best_rerun/mse_5_denormalized,val_best_rerun/mse_5_normalized,val_best_rerun/mse_7_denormalized,val_best_rerun/mse_7_normalized,val_best_rerun/mse_denormalized_mean,val_best_rerun/mse_normalized_mean,val_best_rerun/phm_score_15_denormalized,val_best_rerun/phm_score_15_normalized,val_best_rerun/phm_score_18_denormalized,val_best_rerun/phm_score_18_normalized,val_best_rerun/phm_score_23_denormalized,val_best_rerun/phm_score_23_normalized,val_best_rerun/phm_score_27_denormalized,val_best_rerun/phm_score_27_normalized,val_best_rerun/phm_score_5_denormalized,val_best_rerun/phm_score_5_normalized,val_best_rerun/phm_score_7_denormalized,val_best_rerun/phm_score_7_normalized,val_best_rerun/phm_score_denormalized_mean,val_best_rerun/phm_score_normalized_mean,val_best_rerun/rmse_15_denormalized,val_best_rerun/rmse_15_normalized,val_best_rerun/rmse_18_denormalized,val_best_rerun/rmse_18_normalized,val_best_rerun/rmse_23_denormalized,val_best_rerun/rmse_23_normalized,val_best_rerun/rmse_27_denormalized,val_best_rerun/rmse_27_normalized,val_best_rerun/rmse_5_denormalized,val_best_rerun/rmse_5_normalized,val_best_rerun/rmse_7_denormalized,val_best_rerun/rmse_7_normalized,val_best_rerun/rmse_denormalized_mean,val_best_rerun/rmse_normalized_mean
25,0.005646,44.671867,0.041239,36.080811,0.033308,104.976585,0.096910,92.057945,0.084984,60.715847,0.056051,66.602440,0.061485,67.517586,0.062330,1.934772,1.934772,1.588566,1.588566,3.528280,3.528280,6.192298,6.192297,3.645222,3.645222,3.017918,3.017918,3.317842,3.317842,3270.481201,0.002787,4703.015137,0.004008,14702.636719,0.012530,12116.985352,0.010326,5449.311523,0.004644,6282.075684,0.005354,7754.083984,0.006608,0.356089,0.356089,0.373114,0.373114,0.122099,0.122099,0.190011,0.190011,0.226884,0.226884,0.240738,0.240738,0.251489,0.251489,57.188122,0.052794,68.578537,0.063309,121.254425,0.111937,110.077179,0.101619,73.819450,0.068147,79.259544,0.073169,85.029541,0.078496
26,0.006057,38.695671,0.035722,41.855164,0.038639,122.995995,0.113545,103.576431,0.095618,72.726845,0.067139,62.672249,0.057857,73.753723,0.068087,3.003232,3.003231,2.394802,2.394802,3.774184,3.774184,6.207900,6.207900,5.217603,5.217603,3.536097,3.536097,4.022303,4.022303,2361.650146,0.002013,4285.586426,0.003652,18346.722656,0.015636,14237.776367,0.012134,7600.750488,0.006478,5744.092285,0.004895,8762.763672,0.007468,0.365468,0.365468,0.289375,0.289375,0.049095,0.049095,0.122567,0.122567,0.183412,0.183412,0.244611,0.244611,0.209088,0.209088,48.596813,0.044863,65.464394,0.060434,135.450073,0.125042,119.322151,0.110154,87.182281,0.080483,75.789787,0.069966,88.634254,0.081824
27,0.006139,49.166866,0.045389,46.119526,0.042576,81.977654,0.075679,75.429947,0.069634,62.865887,0.058035,80.776985,0.074

In [8]:
group_runs.sort_values(
    by=["seed", "created_at"] if "created_at" in group_runs else ["seed"]
)[[i for i in group_runs.columns if "test_best_rerun" in i]]

,test_best_rerun/loss,test_best_rerun/mae_16_denormalized,test_best_rerun/mae_16_normalized,test_best_rerun/mae_19_denormalized,test_best_rerun/mae_19_normalized,test_best_rerun/mae_24_denormalized,test_best_rerun/mae_24_normalized,test_best_rerun/mae_28_denormalized,test_best_rerun/mae_28_normalized,test_best_rerun/mae_6_denormalized,test_best_rerun/mae_6_normalized,test_best_rerun/mae_8_denormalized,test_best_rerun/mae_8_normalized,test_best_rerun/mae_denormalized_mean,test_best_rerun/mae_normalized_mean,test_best_rerun/mape_16_denormalized,test_best_rerun/mape_16_normalized,test_best_rerun/mape_19_denormalized,test_best_rerun/mape_19_normalized,test_best_rerun/mape_24_denormalized,test_best_rerun/mape_24_normalized,test_best_rerun/mape_28_denormalized,test_best_rerun/mape_28_normalized,test_best_rerun/mape_6_denormalized,test_best_rerun/mape_6_normalized,test_best_rerun/mape_8_denormalized,test_best_rerun/mape_8_normalized,test_best_rerun/mape_denormalized_mean,test_best_rerun/mape_normalized_mean,test_best_rerun/mse_16_denormalized,test_best_rerun/mse_16_normalized,test_best_rerun/mse_19_denormalized,test_best_rerun/mse_19_normalized,test_best_rerun/mse_24_denormalized,test_best_rerun/mse_24_normalized,test_best_rerun/mse_28_denormalized,test_best_rerun/mse_28_normalized,test_best_rerun/mse_6_denormalized,test_best_rerun/mse_6_normalized,test_best_rerun/mse_8_denormalized,test_best_rerun/mse_8_normalized,test_best_rerun/mse_denormalized_mean,test_best_rerun/mse_normalized_mean,test_best_rerun/phm_score_16_denormalized,test_best_rerun/phm_score_16_normalized,test_best_rerun/phm_score_19_denormalized,test_best_rerun/phm_score_19_normalized,test_best_rerun/phm_score_24_denormalized,test_best_rerun/phm_score_24_normalized,test_best_rerun/phm_score_28_denormalized,test_best_rerun/phm_score_28_normalized,test_best_rerun/phm_score_6_denormalized,test_best_rerun/phm_score_6_normalized,test_best_rerun/phm_score_8_denormalized,test_best_rerun/phm_score_8_normalized,test_best_rerun/phm_score_denormalized_mean,test_best_rerun/phm_score_normalized_mean,test_best_rerun/rmse_16_denormalized,test_best_rerun/rmse_16_normalized,test_best_rerun/rmse_19_denormalized,test_best_rerun/rmse_19_normalized,test_best_rerun/rmse_24_denormalized,test_best_rerun/rmse_24_normalized,test_best_rerun/rmse_28_denormalized,test_best_rerun/rmse_28_normalized,test_best_rerun/rmse_6_denormalized,test_best_rerun/rmse_6_normalized,test_best_rerun/rmse_8_denormalized,test_best_rerun/rmse_8_normalized,test_best_rerun/rmse_denormalized_mean,test_best_rerun/rmse_normalized_mean
25,0.005065,30.792141,0.028426,30.798347,0.028432,87.903671,0.081149,81.827187,0.075540,128.666290,0.118780,59.390087,0.054827,69.896286,0.064526,1.651149,1.651149,4.698690,4.698690,3.989303,3.989303,4.889970,4.889970,5.925451,5.925451,2.817622,2.817622,3.995364,3.995364,1680.807007,0.001432,1605.695190,0.001368,10669.673828,0.009093,9461.178711,0.008063,18545.105469,0.015805,5336.435059,0.004548,7883.149902,0.006718,0.314925,0.314925,0.382616,0.382616,0.168841,0.168841,0.184514,0.184514,0.006502,0.006502,0.269012,0.269012,0.221069,0.221069,40.997646,0.037847,40.071125,0.036992,103.294113,0.095357,97.268585,0.089795,136.180420,0.125717,73.050903,0.067438,81.810463,0.075524
26,0.005970,38.689022,0.035716,33.813931,0.031216,96.416992,0.089008,84.515388,0.078021,156.255219,0.144249,57.150345,0.052759,77.806816,0.071828,2.553461,2.553461,6.677299,6.677299,4.178485,4.178485,4.508988,4.508988,6.977624,6.977624,3.346066,3.346066,4.706987,4.706987,2657.914551,0.002265,1772.029297,0.001510,12418.380859,0.010583,9471.228516,0.008072,27156.468750,0.023144,4989.267090,0.004252,9744.214844,0.008304,0.254609,0.254609,0.307770,0.307770,0.136200,0.136200,0.130625,0.130625,0.001642,0.001642,0.271274,0.271274,0.183687,0.183687,51.554966,0.047594,42.095478,0.038861,111.437790,0.102875,97.320236,0.089842,164.792206,0.152130,70.634743,0.065207,89.639236,0.082752
27,0.004513,26.860533,0.024797,39.791023,0.036734,77